# 03 ? Jalankan pipeline dan simpan satu run lengkap

Menjalankan sel pipeline akan membuat folder run baru. Parameter masalah tetap di `config/v2.yaml`, sedangkan lokasi penyimpanan dan nama submission diatur oleh `config/artifacts.yaml`.

Pipeline menyimpan fitur, semua return core, ambiguitas, hasil video jika tersedia, refinement, diagnostik, submission, dan validasi sebagai CSV/JSON. Core awal dan core setelah video disimpan terpisah. Run gagal memiliki manifest `failed`; run selesai memiliki manifest `completed`.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'games.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.artifacts import list_runs, read_json, read_table


## Dua konfigurasi terpisah

In [2]:
import yaml
from src.pipeline import run_pipeline
MODEL_CONFIG = ROOT / 'config/v2.yaml'
STORAGE_CONFIG = ROOT / 'config/artifacts.yaml'
model_config = yaml.safe_load(MODEL_CONFIG.read_text(encoding='utf-8-sig'))
display(model_config)
display(yaml.safe_load(STORAGE_CONFIG.read_text(encoding='utf-8-sig')))


{'ambiguity': {'margin_threshold': 0.18,
               'max_total_video_frames': 3000,
               'max_windows': 40,
               'window_radius_frames': 35},
 'boundary': {'boundary_shift_cost': 0.01,
              'event_anchor_bonus': 0.12,
              'minimum_quality_for_refine': 0.1,
              'search_radius_frames': 6},
 'control': {'distance_scale_m': 2.5,
             'distance_weight': 1.0,
             'margin_weight': 0.25,
             'persistence_weight': 0.5,
             'proximity_window_frames': 11,
             'tag_weight': 0.3,
             'transition_window_frames': 9},
 'events': {'anchor_window_frames': 25},
 'fps': 25,
 'fusion': {'dead_bias': 0.0,
            'dead_weight': 1.0,
            'live_weight': 1.0,
            'second_half_kickoff_frame': 113742,
            'team_weight': 1.0},
 'game': 'BAR-ATH',
 'live_dead': {'ball_moving_live_weight': 0.22,
               'dead_prior_score': 0.25,
               'live_prior_score': 0.55,
       

{'root_dir': 'artifacts', 'submission_filename': 'baseline_v2.csv'}

## Eksekusi

`RUN_PIPELINE=True` menjalankan pipeline sebenarnya. Gunakan `False` untuk membuka hasil terakhir tanpa menjalankan ulang. Tidak ada logika prediksi atau diagnostik tambahan di cell.

In [3]:
RUN_PIPELINE = True
if RUN_PIPELINE:
    result = run_pipeline(model_config, project_dir=ROOT, storage_config_path=STORAGE_CONFIG,
                          execution_metadata={'notebook': 'notebooks/03_baseline_possession.ipynb',
                                              'model_config_source': str(MODEL_CONFIG)})
    RUN_DIR = Path(result.artifact_dir)
else:
    runs = [r for r in list_runs(project_dir=ROOT) if r['entrypoint'] == 'run_pipeline']
    if not runs:
        raise RuntimeError('Belum ada run selesai.')
    RUN_DIR = Path(runs[-1]['path'])
manifest = read_json(RUN_DIR)
print('Run:', manifest['run_id'])
print('Status:', manifest['status'])
print('Folder:', RUN_DIR)


Run: 20261115_150154_445639_097436f5
Status: completed
Folder: C:\Users\HP 840\Documents\Hizkia ITS\Lomba\MBG\more-than-a-hack-mbg\artifacts\20261115_150154_445639_097436f5


## Hasil yang disimpan

In [4]:
display(read_json(RUN_DIR, 'config/model.json'))
display(read_json(RUN_DIR, 'config/storage.json'))
display(read_json(RUN_DIR, 'submission/write/result.json'))
display(read_json(RUN_DIR, 'submission/validator/result.json'))
display(read_json(RUN_DIR, 'submission/diagnostics.json'))


{'ambiguity': {'margin_threshold': 0.18,
               'max_total_video_frames': 3000,
               'max_windows': 40,
               'window_radius_frames': 35},
 'boundary': {'boundary_shift_cost': 0.01,
              'event_anchor_bonus': 0.12,
              'minimum_quality_for_refine': 0.1,
              'search_radius_frames': 6},
 'control': {'distance_scale_m': 2.5,
             'distance_weight': 1.0,
             'margin_weight': 0.25,
             'persistence_weight': 0.5,
             'proximity_window_frames': 11,
             'tag_weight': 0.3,
             'transition_window_frames': 9},
 'events': {'anchor_window_frames': 25},
 'fps': 25,
 'fusion': {'dead_bias': 0.0,
            'dead_weight': 1.0,
            'live_weight': 1.0,
            'second_half_kickoff_frame': 113742,
            'team_weight': 1.0},
 'game': 'BAR-ATH',
 'live_dead': {'ball_moving_live_weight': 0.22,
               'dead_prior_score': 0.25,
               'live_prior_score': 0.55,
       

{'resolved_root_dir': 'C:\\Users\\HP 840\\Documents\\Hizkia '
                      'ITS\\Lomba\\MBG\\more-than-a-hack-mbg\\artifacts',
 'root_dir': 'artifacts',
 'source_file': 'C:\\Users\\HP 840\\Documents\\Hizkia '
                'ITS\\Lomba\\MBG\\more-than-a-hack-mbg\\config\\artifacts.yaml',
 'submission_filename': 'baseline_v2.csv'}

{'bytes': 1682773,
 'path': 'C:\\Users\\HP 840\\Documents\\Hizkia '
         'ITS\\Lomba\\MBG\\more-than-a-hack-mbg\\artifacts\\20261115_150154_445639_097436f5\\submission\\baseline_v2.csv',
 'rows': 199319,
 'sha256': '04b4ec54fa9ceb03d92f983674e3e832c633916d6b70fe9b28fbc098a25f9022'}

{'message': 'OK    baseline_v2.csv\n'
            '      199319 rows, frame 0..199318\n'
            '\n'
            '      label counts\n'
            '        H    96210   48.3%\n'
            '        A    59713   30.0%\n'
            '        D     2934    1.5%\n'
            '        X    40462   20.3%\n'
            '\n'
            '      reports 1070 possessions of mean 5.8 s\n'
            '      (counted with each X as a gap; the scorer bridges '
            'abstentions inside a real\n'
            '       possession, so its count can be lower than this one)\n'
            '\n'
            'Valid. Note this checks the format only -- it says nothing about '
            'whether the labels are right.\n'
            'Score it against the public ground truth with run_evaluation.py.',
 'status': 'passed'}

{'ball_observed_share_graded': 0.593074274347369,
 'boundaries_examined': 1069,
 'boundaries_shifted': 115,
 'config_sha256': 'cd942c2311157666dc07951563b9c726391ebaeb8a0892e798aac62c2f50bf3e',
 'dead_median_s': 117.36,
 'dead_segments': 1,
 'frame_counts': {'A': 59713, 'D': 2934, 'H': 96210},
 'graded_frames': 158857,
 'possession_mean_s': 5.828897196261682,
 'possession_median_s': 2.04,
 'possession_spells': 1070,
 'preliminary_transitions': 1069,
 'soft_short_spells_merged': 0,
 'submission_sha256': '04b4ec54fa9ceb03d92f983674e3e832c633916d6b70fe9b28fbc098a25f9022',
 'total_transitions': 1069,
 'validator': {'message': 'OK    baseline_v2.csv\n'
                          '      199319 rows, frame 0..199318\n'
                          '\n'
                          '      label counts\n'
                          '        H    96210   48.3%\n'
                          '        A    59713   30.0%\n'
                          '        D     2934    1.5%\n'
                          ' 

## Lanjutkan interpretasi

Buka notebook 01, 02, dan 04. Pilih `RUN_ID` yang dicetak di atas agar seluruh pembahasan merujuk run yang sama. `manifest.json` mencatat daftar CSV/JSON, schema, hash sumber, dan status setiap tahap. Artefak run bersifat lokal dan dikecualikan dari Git.